# L42 â€” AutoEncoder Confusion: Latent Space Distortion Study

**Author:** Hadar Wayn 
**Course:** AI Developer Expert â€” Lesson 42 
**Instructor:** Dr. Yoram Segal 
**Framework:** Keras (TensorFlow backend) 
**GitHub:** [L42-AutoEncoder-Confusion-Keras-Studies](https://github.com/hadarwayn/L42-AutoEncoder-Confusion-Keras-Studies)

---

## What This Notebook Demonstrates

We train AutoEncoders on **one visual domain** and then feed images from a **similar but different domain**. The decoder â€” which only learned one "world" â€” produces fascinating **hallucination distortions**.

| Experiment | Trained On | Confused With | Expected Effect |
|:--|:--|:--|:--|
| **A â€” Faces** | Male faces only | 20 female faces | Female features get masculinized |
| **B â€” Fashion** | Sneakers only | 20 ankle boots | Boot shafts get chopped to sneaker profile |

**Run all cells top-to-bottom** â€” no local setup needed!

## 1. GPU Detection & Hardware Report

This cell checks what GPU Google Colab assigned to us. A GPU makes training ~10x faster. If no GPU is available, the notebook still works â€” just slower.

In [ ]:
# GPU Detection
!nvidia-smi

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    gpu_name = gpus[0].name
    print(f"\n GPU detected: {gpu_name}")
    BATCH_SIZE = 64
    DEVICE = 'GPU'
else:
    print("\n No GPU â€” running on CPU (slower but works fine)")
    BATCH_SIZE = 32
    DEVICE = 'CPU'

print(f"Recommended batch size: {BATCH_SIZE}")

## 2. Install Required Packages

Colab comes with TensorFlow pre-installed, but we need a few extra libraries for visualization and data handling.

In [ ]:
!pip install -q scikit-learn matplotlib numpy pillow tqdm scipy

## 3. Google Drive Mount (Optional)

Mount Google Drive to save results permanently. If you skip this, results are lost when the Colab session ends.

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/L42_results'
    os.makedirs(SAVE_DIR, exist_ok=True)
    print(f"Results will be saved to: {SAVE_DIR}")
except ImportError:
    SAVE_DIR = '/content/L42_results'
    os.makedirs(SAVE_DIR, exist_ok=True)
    print(f"Not on Colab â€” saving to: {SAVE_DIR}")

## 4. How AutoEncoders Work â€” The Artist Analogy

Imagine training an artist to paint **only men's faces** â€” thousands of them. The artist learns every detail: strong jawlines, facial hair patterns, broader skull proportions.

Now you show the artist a **woman's photo** and ask them to redraw it. The artist has **never learned** how to draw smooth female jawlines or delicate features. So they do their best â€” they draw the general face shape correctly, but the details come out *mannish*.

That's exactly what happens inside an AutoEncoder's decoder. It can only reconstruct what it was taught.

### The Three Parts of an AutoEncoder

```
Original Image  â†’  [ENCODER]  â†’  Compressed Code  â†’  [DECODER]  â†’  Reconstructed Image
   (input)         (compresses)    (bottleneck)       (rebuilds)      (output)
```

- **Encoder**: Squeezes the image into a small numerical code (like summarizing a book into a tweet)
- **Bottleneck**: The compressed representation â€” forces the network to learn only the most important features
- **Decoder**: Rebuilds the image from the code â€” but can only use patterns it learned during training

## 5. Imports & Configuration

Load all libraries and define the configuration for both experiments.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import time

import keras
from keras import layers, Model
from keras.datasets import fashion_mnist
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

SEED = 42
DPI = 300

CONFIG_FASHION = {
    'input_shape': (28, 28, 1),
    'filters': [64, 128, 256],
    'latent_dim': 32,
    'loss': 'binary_crossentropy',
    'learning_rate': 0.001,
}

CONFIG_FACES = {
    'input_shape': (64, 64, 3),
    'filters': [64, 128, 256],
    'latent_dim': 32,
    'loss': 'mse',
    'learning_rate': 0.001,
}

EPOCHS_FASHION = 50
EPOCHS_FACES = 50

print('All imports successful!')
print(f'Keras version: {keras.__version__}')
print(f'NumPy version: {np.__version__}')

## 6. AutoEncoder Architecture

This is the core neural network. It uses double-conv blocks with LeakyReLU(0.2) for improved gradient flow:

`Input → Conv2x(64) → Conv2x(128) → Conv2x(256) → Bottleneck → Conv2x(256) → Conv2x(128) → Conv2x(64) → Output`

**Key design choices**: Double convolution blocks per resolution level provide richer feature extraction, LeakyReLU(0.2) prevents dead neurons, and BatchNormalization after every convolutional layer prevents the blurry outputs that plagued the reference projects.

In [ ]:
def _enc_block(x, filters, idx):
    x = layers.Conv2D(filters, (3, 3), padding='same', name=f'enc_conv_{idx}a')(x)
    x = layers.BatchNormalization(name=f'enc_bn_{idx}a')(x)
    x = layers.LeakyReLU(0.2, name=f'enc_lrelu_{idx}a')(x)
    x = layers.Conv2D(filters, (3, 3), padding='same', name=f'enc_conv_{idx}b')(x)
    x = layers.BatchNormalization(name=f'enc_bn_{idx}b')(x)
    x = layers.LeakyReLU(0.2, name=f'enc_lrelu_{idx}b')(x)
    x = layers.MaxPooling2D((2, 2), name=f'enc_pool_{idx}')(x)
    return x

def _dec_block(x, filters, idx):
    x = layers.UpSampling2D((2, 2), name=f'dec_up_{idx}')(x)
    x = layers.Conv2DTranspose(filters, (3, 3), padding='same', name=f'dec_conv_{idx}a')(x)
    x = layers.BatchNormalization(name=f'dec_bn_{idx}a')(x)
    x = layers.LeakyReLU(0.2, name=f'dec_lrelu_{idx}a')(x)
    x = layers.Conv2DTranspose(filters, (3, 3), padding='same', name=f'dec_conv_{idx}b')(x)
    x = layers.BatchNormalization(name=f'dec_bn_{idx}b')(x)
    x = layers.LeakyReLU(0.2, name=f'dec_lrelu_{idx}b')(x)
    return x

def build_autoencoder(config):
    input_shape = tuple(config['input_shape'])
    filters = config['filters']
    latent_dim = config['latent_dim']

    encoder_input = layers.Input(shape=input_shape, name='encoder_input')
    x = encoder_input
    for i, f in enumerate(filters):
        x = _enc_block(x, f, i)

    shape_before_flatten = x.shape[1:]
    x = layers.Flatten(name='enc_flatten')(x)
    latent = layers.Dense(latent_dim, name='latent_vector')(x)
    encoder = Model(encoder_input, latent, name='encoder')

    flat_dim = int(np.prod(shape_before_flatten))
    decoder_input = layers.Input(shape=(latent_dim,), name='decoder_input')
    x = layers.Dense(flat_dim, name='dec_dense')(decoder_input)
    x = layers.Reshape(shape_before_flatten, name='dec_reshape')(x)
    for i, f in enumerate(reversed(filters)):
        x = _dec_block(x, f, i)

    x = layers.Conv2DTranspose(input_shape[-1], (3, 3), padding='same', activation='sigmoid', name='dec_output')(x)
    x = layers.Resizing(input_shape[0], input_shape[1], name='dec_resize')(x)
    decoder = Model(decoder_input, x, name='decoder')

    encoded = encoder(encoder_input)
    decoded = decoder(encoded)
    autoencoder = Model(encoder_input, decoded, name='autoencoder')
    autoencoder.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config['learning_rate']),
        loss=config['loss'],
    )
    print(f"AutoEncoder built — Input: {input_shape} | Latent: {latent_dim} | Params: {autoencoder.count_params():,}")
    return {'autoencoder': autoencoder, 'encoder': encoder, 'decoder': decoder}

# Quick validation
test_models = build_autoencoder(CONFIG_FASHION)
dummy = np.zeros((1, 28, 28, 1), dtype=np.float32)
out = test_models['autoencoder'].predict(dummy, verbose=0)
print(f'Validation: input {dummy.shape} → output {out.shape} ✓')
del test_models, dummy, out

## 7. Visualization Functions

These functions create all the plots we need to tell the story of how the AutoEncoder gets confused. Every plot includes a plain-English explanation.

In [ ]:
def save_and_show(fig, path):
    """Save figure at 300 DPI and display inline."""
    fig.savefig(path, dpi=DPI, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)
    print(f'Saved: {path}')


def plot_convergence(history, name, path):
    """Training vs validation loss over epochs."""
    fig, ax = plt.subplots(figsize=(10, 6))
    epochs = range(1, len(history['train_loss']) + 1)
    ax.plot(epochs, history['train_loss'], 'b-', label='Training Loss', linewidth=2)
    ax.plot(epochs, history['val_loss'], 'r-', label='Validation Loss', linewidth=2)
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Loss', fontsize=12)
    ax.set_title(f'AutoEncoder Learning Curve â€” {name}', fontsize=14)
    fig.text(0.5, 0.01, 'Loss decreasing = network is learning to reconstruct images',
             ha='center', fontsize=10, style='italic', color='gray')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    save_and_show(fig, path)


def plot_in_distribution(originals, reconstructed, name, path):
    """5 in-distribution pairs: original vs reconstructed."""
    n = min(5, len(originals))
    is_gray = originals.shape[-1] == 1
    fig, axes = plt.subplots(2, n, figsize=(3 * n, 6))
    fig.suptitle(f'Network Works Correctly on Training Domain â€” {name}',
                 fontsize=13, fontweight='bold')
    fig.text(0.5, 0.92, 'These images are from the training domain â€” reconstructed accurately',
             ha='center', fontsize=10, style='italic', color='gray')
    for i in range(n):
        img_o = originals[i].squeeze() if is_gray else originals[i]
        img_r = reconstructed[i].squeeze() if is_gray else reconstructed[i]
        cmap = 'gray' if is_gray else None
        axes[0, i].imshow(img_o, cmap=cmap, vmin=0, vmax=1)
        axes[0, i].set_title('Original', fontsize=9); axes[0, i].axis('off')
        axes[1, i].imshow(img_r, cmap=cmap, vmin=0, vmax=1)
        mse = float(np.mean((originals[i] - reconstructed[i]) ** 2))
        axes[1, i].set_title(f'Reconstructed\nMSE: {mse:.4f}', fontsize=9)
        axes[1, i].axis('off')
    fig.tight_layout(rect=[0, 0.02, 1, 0.90])
    save_and_show(fig, path)


def plot_confusion_grid(originals, reconstructed, diff_maps, mse_list, name, path):
    """20-row confusion grid: Original | Reconstructed | Difference Map."""
    n = len(originals)
    is_gray = originals.shape[-1] == 1
    fig, axes = plt.subplots(n, 3, figsize=(12, 2.5 * n))
    fig.suptitle(f'The Confusion Effect â€” {name}', fontsize=14, fontweight='bold', y=1.0)
    col_titles = ['Original Input (OoD)', 'Reconstructed Output', 'Distortion Map']
    for j, title in enumerate(col_titles):
        axes[0, j].set_title(title, fontsize=11, fontweight='bold')
    for i in range(n):
        img_o = originals[i].squeeze() if is_gray else originals[i]
        img_r = reconstructed[i].squeeze() if is_gray else reconstructed[i]
        diff = diff_maps[i].squeeze() if is_gray else np.mean(diff_maps[i], axis=-1)
        cmap = 'gray' if is_gray else None
        axes[i, 0].imshow(img_o, cmap=cmap, vmin=0, vmax=1)
        axes[i, 0].set_ylabel(f'#{i+1}', fontsize=9, rotation=0, labelpad=25)
        axes[i, 0].set_xticks([]); axes[i, 0].set_yticks([])
        axes[i, 1].imshow(img_r, cmap=cmap, vmin=0, vmax=1)
        if i > 0: axes[i, 1].set_title(f'MSE: {mse_list[i]:.4f}', fontsize=8)
        axes[i, 1].axis('off')
        axes[i, 2].imshow(diff, cmap='hot', vmin=0, vmax=diff.max() + 1e-8)
        axes[i, 2].axis('off')
    fig.tight_layout()
    save_and_show(fig, path)


def plot_distortion_highlights(originals, reconstructed, mse_list, name, path):
    """Top 5 most distorted images."""
    top5_idx = np.argsort(mse_list)[-5:][::-1]
    is_gray = originals.shape[-1] == 1
    fig, axes = plt.subplots(5, 3, figsize=(12, 14))
    fig.suptitle(f'Top 5 Most Confused Reconstructions â€” {name}', fontsize=14, fontweight='bold')
    for row, idx in enumerate(top5_idx):
        img_o = originals[idx].squeeze() if is_gray else originals[idx]
        img_r = reconstructed[idx].squeeze() if is_gray else reconstructed[idx]
        diff = np.abs(originals[idx] - reconstructed[idx])
        diff_show = diff.squeeze() if is_gray else np.mean(diff, axis=-1)
        cmap = 'gray' if is_gray else None
        axes[row, 0].imshow(img_o, cmap=cmap, vmin=0, vmax=1)
        axes[row, 0].set_title('Original' if row == 0 else '', fontsize=10)
        axes[row, 0].set_ylabel(f'MSE: {mse_list[idx]:.4f}', fontsize=9)
        axes[row, 0].set_xticks([]); axes[row, 0].set_yticks([])
        axes[row, 1].imshow(img_r, cmap=cmap, vmin=0, vmax=1)
        axes[row, 1].set_title('Reconstructed' if row == 0 else '', fontsize=10)
        axes[row, 1].axis('off')
        axes[row, 2].imshow(diff_show, cmap='hot')
        axes[row, 2].set_title('Distortion Map' if row == 0 else '', fontsize=10)
        axes[row, 2].axis('off')
    fig.tight_layout()
    save_and_show(fig, path)


def plot_error_distribution(in_dist_mse, ood_mse, name, path):
    """Overlapping histograms: in-distribution vs OoD MSE."""
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(in_dist_mse, bins=15, alpha=0.6, color='blue',
            label=f'In-Distribution (mean={np.mean(in_dist_mse):.4f})')
    ax.hist(ood_mse, bins=15, alpha=0.6, color='red',
            label=f'OoD / Confused (mean={np.mean(ood_mse):.4f})')
    ax.axvline(np.mean(in_dist_mse), color='blue', linestyle='--', linewidth=2)
    ax.axvline(np.mean(ood_mse), color='red', linestyle='--', linewidth=2)
    ax.set_xlabel('Reconstruction MSE', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title(f'Reconstruction Error: Normal vs Confused â€” {name}', fontsize=14)
    fig.text(0.5, 0.01, 'OoD images have higher error â€” the network struggles with unfamiliar input',
             ha='center', fontsize=10, style='italic', color='gray')
    ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
    save_and_show(fig, path)


def plot_latent_pca(encoder, x_in, x_ood, name, path, max_in=500):
    """PCA projection of latent space â€” in-dist (blue) vs OoD (red)."""
    if len(x_in) > max_in:
        rng = np.random.RandomState(42)
        x_in = x_in[rng.choice(len(x_in), max_in, replace=False)]
    lat_in = encoder.predict(x_in, verbose=0)
    lat_ood = encoder.predict(x_ood, verbose=0)
    all_lat = np.concatenate([lat_in, lat_ood], axis=0)
    pca = PCA(n_components=2, random_state=42)
    all_2d = pca.fit_transform(all_lat)
    in_2d, ood_2d = all_2d[:len(lat_in)], all_2d[len(lat_in):]

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.scatter(in_2d[:, 0], in_2d[:, 1], c='blue', alpha=0.3, s=20,
               label=f'In-Distribution ({len(in_2d)})')
    ax.scatter(ood_2d[:, 0], ood_2d[:, 1], c='red', marker='*', s=150,
               edgecolors='darkred', linewidths=0.5,
               label=f'OoD â€” Confused ({len(ood_2d)})', zorder=5)
    # Convex hulls
    try:
        from scipy.spatial import ConvexHull
        for pts, color in [(in_2d, 'blue'), (ood_2d, 'red')]:
            if len(pts) >= 3:
                hull = ConvexHull(pts)
                hp = np.append(hull.vertices, hull.vertices[0])
                ax.fill(pts[hp, 0], pts[hp, 1], color=color, alpha=0.08 if color == 'blue' else 0.15)
    except Exception: pass
    ax.set_xlabel(f'PCA 1 ({pca.explained_variance_ratio_[0]:.1%})', fontsize=11)
    ax.set_ylabel(f'PCA 2 ({pca.explained_variance_ratio_[1]:.1%})', fontsize=11)
    ax.set_title(f'Latent Space: Where OoD Images Land â€” {name}', fontsize=14, fontweight='bold')
    fig.text(0.5, 0.01, 'Red stars = OoD inputs forced into wrong region of latent space',
             ha='center', fontsize=10, style='italic', color='gray')
    ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
    save_and_show(fig, path)

print('Visualization functions defined âœ“')

## 8. Helper Functions

Training and confusion (OoD inference) pipeline functions.

In [ ]:
def train_model(autoencoder, x_train, x_val, epochs, batch_size, save_prefix):
    """Train autoencoder with checkpointing and early stopping."""
    os.makedirs(save_prefix, exist_ok=True)
    best_path = os.path.join(save_prefix, 'best_weights.keras')

    callbacks = [
        ModelCheckpoint(best_path, monitor='val_loss', save_best_only=True, verbose=1),
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    ]

    start = time.time()
    history = autoencoder.fit(
        x_train, x_train,
        validation_data=(x_val, x_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=1,
    )
    train_time = time.time() - start

    result = {
        'train_loss': history.history['loss'],
        'val_loss': history.history['val_loss'],
        'epochs_run': len(history.history['loss']),
        'training_time_sec': train_time,
    }
    print(f"Training complete â€” val_loss: {result['val_loss'][-1]:.6f} | "
          f"Epochs: {result['epochs_run']} | Time: {train_time:.1f}s")
    return result


def run_confusion(autoencoder, x_ood):
    """Feed OoD images through autoencoder, compute reconstruction error."""
    reconstructed = np.clip(autoencoder.predict(x_ood, verbose=0), 0.0, 1.0)
    diff_maps = np.abs(x_ood - reconstructed)
    mse_per_image = np.mean((x_ood - reconstructed) ** 2, axis=(1, 2, 3))
    mean_mse = float(np.mean(mse_per_image))
    print(f"Confusion: {len(x_ood)} images | Mean MSE: {mean_mse:.6f}")
    return {'originals': x_ood, 'reconstructed': reconstructed,
            'diff_maps': diff_maps, 'mse_per_image': mse_per_image, 'mean_mse': mean_mse}

print('Helper functions defined âœ“')

---

# EXPERIMENT B â€” Fashion-MNIST: Sneakers vs Ankle Boots

We train the AutoEncoder on **sneakers only** (Fashion-MNIST class 7). Then we feed it **20 ankle boots** (class 9) and watch the decoder try to compress tall boots into low sneaker profiles.

**What to expect:** The boot shafts get "chopped off" â€” the decoder doesn't know how to draw anything taller than a sneaker.

### B.1 â€” Load Fashion-MNIST Data

Fashion-MNIST is built into Keras â€” no manual download needed. We extract sneakers for training and boots for confusion testing.

In [ ]:
# Load Fashion-MNIST
(x_all_train, y_all_train), (x_all_test, y_all_test) = fashion_mnist.load_data()
x_all = np.concatenate([x_all_train, x_all_test], axis=0)
y_all = np.concatenate([y_all_train, y_all_test], axis=0)

sneakers = x_all[y_all == 7].astype(np.float32) / 255.0
boots = x_all[y_all == 9].astype(np.float32) / 255.0
sneakers = sneakers[..., np.newaxis]  # (N, 28, 28, 1)
boots = boots[..., np.newaxis]

x_train_b, x_val_b = train_test_split(sneakers, test_size=0.15, random_state=SEED)
rng = np.random.RandomState(SEED)
x_test_b = boots[rng.choice(len(boots), 20, replace=False)]

print(f'Training: {len(x_train_b)} sneakers')
print(f'Validation: {len(x_val_b)} sneakers')
print(f'OoD Test: {len(x_test_b)} ankle boots')
print(f'Shape: {x_train_b.shape} | Range: [{x_train_b.min():.1f}, {x_train_b.max():.1f}]')

### B.2 â€” Build & Train AutoEncoder on Sneakers

The network learns to compress and reconstruct sneaker images. It has never seen a boot â€” that's the whole point.

In [ ]:
models_b = build_autoencoder(CONFIG_FASHION)
save_dir_b = os.path.join(SAVE_DIR, 'experiment_b')

history_b = train_model(
    models_b['autoencoder'], x_train_b, x_val_b,
    epochs=EPOCHS_FASHION, batch_size=BATCH_SIZE,
    save_prefix=os.path.join(save_dir_b, 'model_weights')
)

### B.3 â€” Confusion Phase: Feed Boots to Sneaker-Trained Network

Now the magic happens. We feed 20 ankle boot images into the network that only knows sneakers. The decoder will try to reconstruct them as sneakers â€” chopping off the boot shaft.

In [ ]:
# OoD confusion (boots through sneaker model)
ood_b = run_confusion(models_b['autoencoder'], x_test_b)

# In-distribution baseline (sneakers through sneaker model)
in_dist_indices = np.random.RandomState(42).choice(len(x_val_b), 20, replace=False)
in_dist_b = run_confusion(models_b['autoencoder'], x_val_b[in_dist_indices])

ratio_b = ood_b['mean_mse'] / in_dist_b['mean_mse']
print(f'\nIn-dist MSE: {in_dist_b["mean_mse"]:.6f}')
print(f'OoD MSE:     {ood_b["mean_mse"]:.6f}')
print(f'Ratio:       {ratio_b:.2f}x (higher = more confusion)')

### B.4 â€” Visualizations: Experiment B Results

Six plots that tell the complete story:
1. **Convergence** â€” did the network learn?
2. **In-Distribution** â€” does it work on sneakers?
3. **Confusion Grid** â€” what happens with boots?
4. **Top 5 Distortions** â€” the most dramatic failures
5. **Error Distribution** â€” statistical proof of confusion
6. **Latent Space PCA** â€” geometric proof of confusion

In [ ]:
os.makedirs(save_dir_b, exist_ok=True)
NAME_B = 'Fashion-MNIST (Sneakers vs Boots)'

plot_convergence(history_b, NAME_B, os.path.join(save_dir_b, 'convergence_plot.png'))

In [ ]:
plot_in_distribution(in_dist_b['originals'], in_dist_b['reconstructed'],
                     NAME_B, os.path.join(save_dir_b, 'in_distribution_reconstruction.png'))

In [ ]:
plot_confusion_grid(ood_b['originals'], ood_b['reconstructed'],
                    ood_b['diff_maps'], ood_b['mse_per_image'],
                    NAME_B, os.path.join(save_dir_b, 'ood_confusion_grid.png'))

In [ ]:
plot_distortion_highlights(ood_b['originals'], ood_b['reconstructed'],
                           ood_b['mse_per_image'], NAME_B,
                           os.path.join(save_dir_b, 'distortion_highlight.png'))

In [ ]:
plot_error_distribution(in_dist_b['mse_per_image'], ood_b['mse_per_image'],
                        NAME_B, os.path.join(save_dir_b, 'error_distribution.png'))

In [ ]:
plot_latent_pca(models_b['encoder'], x_val_b, x_test_b,
                NAME_B, os.path.join(save_dir_b, 'latent_space_pca.png'))

---

# EXPERIMENT A â€” Human Faces: Male-Trained vs Female-Tested

We train the AutoEncoder on **male faces only**. Then we feed it **20 female faces** and observe how the decoder imposes male facial features (stronger jawlines, broader proportions) onto the female inputs.

**Dataset:** LFW (Labeled Faces in the Wild) â€” automatically downloaded by sklearn. Gender split is based on known public figure names.

### A.1 â€” Load Face Data

We use the LFW face dataset (sklearn auto-download). Male faces are used for training, 20 female faces for confusion testing.

In [ ]:
from sklearn.datasets import fetch_lfw_people
from PIL import Image

print('Downloading LFW face dataset...')
lfw = fetch_lfw_people(min_faces_per_person=20, resize=1.0)

# Known female names in LFW
female_names = {
    'Amelie Mauresmo', 'Angelina Jolie', 'Gloria Macapagal Arroyo',
    'Jennifer Aniston', 'Jennifer Capriati', 'Jennifer Lopez',
    'Laura Bush', 'Lindsay Davenport', 'Megawati Sukarnoputri',
    'Naomi Watts', 'Serena Williams', 'Winona Ryder',
}

female_idx, male_idx = [], []
for i, target in enumerate(lfw.target):
    name = lfw.target_names[target]
    (female_idx if name in female_names else male_idx).append(i)

print(f'Males: {len(male_idx)} | Females: {len(female_idx)}')

# Convert LFW grayscale to 64x64 RGB
def resize_lfw(images, size=(64, 64)):
    result = []
    for img in images:
        pil = Image.fromarray((img * 255).astype(np.uint8), mode='L')
        pil = pil.convert('RGB').resize(size, Image.LANCZOS)
        result.append(np.array(pil))
    return np.stack(result).astype(np.float32) / 255.0

male_images = resize_lfw(lfw.images[male_idx])
female_images = resize_lfw(lfw.images[female_idx])

x_train_a, x_val_a = train_test_split(male_images, test_size=0.15, random_state=SEED)
rng_a = np.random.RandomState(SEED)
x_test_a = female_images[rng_a.choice(len(female_images), 20, replace=False)]

print(f'Training: {len(x_train_a)} male faces')
print(f'Validation: {len(x_val_a)} male faces')
print(f'OoD Test: {len(x_test_a)} female faces')
print(f'Shape: {x_train_a.shape}')

### A.2 â€” Build & Train AutoEncoder on Male Faces

The network learns to reconstruct male faces. It has never seen a female face during training.

In [ ]:
models_a = build_autoencoder(CONFIG_FACES)
save_dir_a = os.path.join(SAVE_DIR, 'experiment_a')

history_a = train_model(
    models_a['autoencoder'], x_train_a, x_val_a,
    epochs=EPOCHS_FACES, batch_size=BATCH_SIZE,
    save_prefix=os.path.join(save_dir_a, 'model_weights')
)

### A.3 â€” Confusion Phase: Feed Female Faces to Male-Trained Network

The decoder only knows male features. When it tries to reconstruct female faces, it imposes its male-learned patterns: squared jawlines, broader proportions, rougher textures.

In [ ]:
ood_a = run_confusion(models_a['autoencoder'], x_test_a)

in_dist_indices_a = np.random.RandomState(42).choice(len(x_val_a), 20, replace=False)
in_dist_a = run_confusion(models_a['autoencoder'], x_val_a[in_dist_indices_a])

ratio_a = ood_a['mean_mse'] / in_dist_a['mean_mse']
print(f'\nIn-dist MSE: {in_dist_a["mean_mse"]:.6f}')
print(f'OoD MSE:     {ood_a["mean_mse"]:.6f}')
print(f'Ratio:       {ratio_a:.2f}x')

### A.4 â€” Visualizations: Experiment A Results

In [ ]:
os.makedirs(save_dir_a, exist_ok=True)
NAME_A = 'Human Faces (Male-Trained vs Female-Tested)'

plot_convergence(history_a, NAME_A, os.path.join(save_dir_a, 'convergence_plot.png'))

In [ ]:
plot_in_distribution(in_dist_a['originals'], in_dist_a['reconstructed'],
                     NAME_A, os.path.join(save_dir_a, 'in_distribution_reconstruction.png'))

In [ ]:
plot_confusion_grid(ood_a['originals'], ood_a['reconstructed'],
                    ood_a['diff_maps'], ood_a['mse_per_image'],
                    NAME_A, os.path.join(save_dir_a, 'ood_confusion_grid.png'))

In [ ]:
plot_distortion_highlights(ood_a['originals'], ood_a['reconstructed'],
                           ood_a['mse_per_image'], NAME_A,
                           os.path.join(save_dir_a, 'distortion_highlight.png'))

In [ ]:
plot_error_distribution(in_dist_a['mse_per_image'], ood_a['mse_per_image'],
                        NAME_A, os.path.join(save_dir_a, 'error_distribution.png'))

In [ ]:
plot_latent_pca(models_a['encoder'], x_val_a, x_test_a,
                NAME_A, os.path.join(save_dir_a, 'latent_space_pca.png'))

---

## Final Summary

Training time comparison and key metrics from both experiments.

In [ ]:
print('=' * 70)
print('FINAL SUMMARY â€” L42 AutoEncoder Confusion Study')
print('=' * 70)
print()
print(f'Hardware: {DEVICE} | Batch size: {BATCH_SIZE}')
print()
print(f'{"Metric":<35} {"Experiment B (Fashion)":>22} {"Experiment A (Faces)":>22}')
print('-' * 80)
print(f'{"Training epochs":.<35} {history_b["epochs_run"]:>22} {history_a["epochs_run"]:>22}')
print(f'{"Training time (sec)":.<35} {history_b["training_time_sec"]:>22.1f} {history_a["training_time_sec"]:>22.1f}')
print(f'{"Final val_loss":.<35} {history_b["val_loss"][-1]:>22.6f} {history_a["val_loss"][-1]:>22.6f}')
print(f'{"In-dist MSE":.<35} {in_dist_b["mean_mse"]:>22.6f} {in_dist_a["mean_mse"]:>22.6f}')
print(f'{"OoD MSE":.<35} {ood_b["mean_mse"]:>22.6f} {ood_a["mean_mse"]:>22.6f}')
print(f'{"Confusion ratio (OoD/In-dist)":.<35} {ratio_b:>22.2f}x {ratio_a:>22.2f}x')
print()
print(f'All results saved to: {SAVE_DIR}')
print()
print('Key Insight: Both experiments show measurably higher reconstruction')
print('error for OoD images â€” proving the decoder hallucinates training-domain')
print('features when confronted with unfamiliar input.')

---

## What We Learned

1. **AutoEncoders only know their training world** â€” the decoder reconstructs everything using patterns from training data only
2. **Structural hallucination is measurable** â€” OoD reconstruction MSE is consistently higher than in-distribution MSE
3. **Latent space geometry matters** â€” PCA shows OoD inputs landing in different regions of the latent space
4. **BatchNormalization is critical** â€” without it, outputs are blurry gray blobs (as seen in reference projects)
5. **This principle powers real applications** â€” anomaly detection, medical imaging, quality control all use this exact pattern

---

*Notebook by Hadar Wayn â€” AI Developer Expert Course, Lesson 42*